In [11]:
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import xgboost as xgb
import re
from sklearn.metrics import r2_score, mean_squared_error
import joblib







df_train=pd.read_csv(r"G:\My Drive\amazon ml\train.csv")






def smape_metric(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2

    ratio = np.where(denominator == 0, 0, numerator / denominator)

    return np.mean(ratio) * 100
TRAIN_SIZE = 55000
df_train = df_train.sample(n=TRAIN_SIZE, random_state=42).reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    return text

df_train['catalog_content'] = df_train['catalog_content'].apply(clean_text)

# ---------- REMOVE OUTLIERS ----------
train = df_train[df_train['price'] < df_train['price'].quantile(0.99)].reset_index(drop=True)

# ---------- FEATURE ENGINEERING ----------
train['word_count'] = train['catalog_content'].apply(lambda x: len(str(x).split()))
train['char_count'] = train['catalog_content'].apply(lambda x: len(str(x)))
train['digit_count'] = train['catalog_content'].apply(lambda x: sum(c.isdigit() for c in str(x)))

# ---------- TF-IDF + EXTRA FEATURES ----------
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_text_train = tfidf.fit_transform(train['catalog_content'].astype(str))
X_extra_train = sparse.csr_matrix(train[['word_count','char_count','digit_count']].values)
X_combined_train = sparse.hstack([X_text_train,X_extra_train])

y_train = np.log1p(train['price'])

# ---------- TRAIN-VALIDATION SPLIT ----------
X_train_final,X_valid_final,y_train_final,y_valid_final = train_test_split(
    X_combined_train,y_train,test_size=0.2,random_state=42
)




'''def objective(trial):

    params = {
        'tree_method': 'hist',
        'device': 'cuda',
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
   #     'n_estimators': trial.suggest_int('n_estimators', 200, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42,
        'max_bin': trial.suggest_categorical('max_bin', [256, 300]),
    }

    # 2. Train the model with the suggested params
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=600, # Max number of boosting rounds
        evals=[(dvalid, 'validation')],
        early_stopping_rounds=30,
        verbose_eval=False # Suppress round-by-round output
    )
    # 3. Evaluate and return the score to be minimized
    preds_log = model.predict(dvalid)
    mse = mean_squared_error(y_valid_final, preds_log)
    rmse=np.sqrt(mse)

    y_valid_original = np.expm1(y_valid_final)
    preds_original = np.expm1(preds_log)

    smape = smape_metric(y_valid_original, preds_original)
    trial.set_user_attr("smape", smape)

    return rmse
# Create a study object and specify direction is to minimize the metric
study = optuna.create_study(direction='minimize')

# Start the optimization!
study.optimize(objective, n_trials=70)

# --- RESULTS ---
print("Best trial:")
trial = study.best_trial
print(f"  Value (RMSE): {trial.value}")
print("  Best hyperparameters: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")'''

'def objective(trial):\n\n    params = {\n        \'tree_method\': \'hist\',\n        \'device\': \'cuda\',\n        \'objective\': \'reg:squarederror\',\n        \'eval_metric\': \'rmse\',\n   #     \'n_estimators\': trial.suggest_int(\'n_estimators\', 200, 1000, step=100),\n        \'learning_rate\': trial.suggest_float(\'learning_rate\', 0.01, 0.3, log=True),\n        \'max_depth\': trial.suggest_int(\'max_depth\', 4, 12),\n        \'subsample\': trial.suggest_float(\'subsample\', 0.6, 1.0),\n        \'colsample_bytree\': trial.suggest_float(\'colsample_bytree\', 0.6, 1.0),\n        \'gamma\': trial.suggest_float(\'gamma\', 1e-8, 1.0, log=True),\n        \'min_child_weight\': trial.suggest_int(\'min_child_weight\', 1, 10),\n        \'random_state\': 42,\n        \'max_bin\': trial.suggest_categorical(\'max_bin\', [256, 300]),\n    }\n\n    # 2. Train the model with the suggested params\n    model = xgb.train(\n        params=params,\n        dtrain=dtrain,\n        num_boost_round=6

In [12]:
joblib.dump(tfidf,"bishal.pkl")

['bishal.pkl']

In [3]:
from xgboost import XGBRegressor
model = XGBRegressor(
    learning_rate=0.055665,
    max_depth=12,
    subsample=0.9657,
    colsample_bytree=0.838,
    gamma=0.22,
    min_child_weight=9,
    max_bin=300,
    n_estimators=500,
    objective='reg:squarederror',
    random_state=42
)

In [4]:
model.fit(X_train_final, y_train_final)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.838, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.22, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.055665, max_bin=300, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=12,
             max_leaves=None, min_child_weight=9, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [10]:
import joblib
joblib.dump(model, "final_updated_shaurya_bishal_model.pkl")

['final_updated_shaurya_bishal_model.pkl']

In [14]:
from xgboost import booster

ImportError: cannot import name 'booster' from 'xgboost' (C:\Users\bisha\AppData\Roaming\Python\Python312\site-packages\xgboost\__init__.py)

In [10]:
import joblib
joblib.dump(model, "final_updated_shaurya_bishal_model.pkl")

['final_updated_shaurya_bishal_model.pkl']

In [5]:
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0  # avoid divide by zero
    return np.mean(diff) * 100

In [6]:
y_pred=model.predict(X_valid_final)

In [7]:
smape_score = smape(y_valid_final, y_pred)

print(f"Validation SMAPE: {smape_score:.4f}")

Validation SMAPE: 21.1165


In [9]:
y_pred=model.predict(X_train_final)
smape_score=smape(y_train_final, y_pred)

print(f"Validation SMAPE: {smape_score:.4f}")

Validation SMAPE: 15.9146
